## Silver validation logic ##

In [0]:

from pyspark.sql import functions as F

# Read the Bronze Delta table
bronze_df = spark.table(
    "workspace.bronze.yellow_taxi_trips"
)

# Create validation and derived columns
silver_staged = (
    bronze_df

    # Calculate trip duration in minutes
    .withColumn(
        "trip_duration_minutes",
        F.expr(
            "timestampdiff(SECOND, "
            "tpep_pickup_datetime, "
            "tpep_dropoff_datetime)"
        ) / 60.0
    )

    # Identify missing timestamps
    .withColumn(
        "is_missing_timestamp",
        F.col("tpep_pickup_datetime").isNull() |
        F.col("tpep_dropoff_datetime").isNull()
    )

    # Identify negative trip duration
    .withColumn(
        "is_negative_duration",
        F.col("tpep_dropoff_datetime") <
        F.col("tpep_pickup_datetime")
    )

    # Flag zero-distance trips
    .withColumn(
        "is_zero_distance",
        F.col("trip_distance") == 0
    )

    # Flag negative fares
    .withColumn(
        "is_negative_fare",
        F.col("fare_amount") < 0
    )

    # Classify records using initial critical rules
    .withColumn(
        "quality_status",
        F.when(
            F.col("is_missing_timestamp") |
            F.col("is_negative_duration"),
            F.lit("QUARANTINE")
        ).otherwise(F.lit("ACCEPT"))
    )
)

display(
    silver_staged.groupBy("quality_status")
    .count()
)


## Save the results as permanent Silver Delta tables ##

In [0]:

from pyspark.sql import functions as F

# Create the Silver schema
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

# Explain why each rejected record failed
silver_staged = silver_staged.withColumn(
    "quality_reason",
    F.when(
        F.col("is_missing_timestamp"),
        "MISSING_TIMESTAMP"
    ).when(
        F.col("is_negative_duration"),
        "NEGATIVE_DURATION"
    ).otherwise(F.lit(None).cast("string"))
)

# Separate accepted and quarantined trips
silver_valid = silver_staged.filter(
    F.col("quality_status") == "ACCEPT"
)

silver_quarantine = silver_staged.filter(
    F.col("quality_status") == "QUARANTINE"
)

# Save accepted trips
(
    silver_valid.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.yellow_taxi_trips")
)

# Save rejected trips
(
    silver_quarantine.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.yellow_taxi_quarantine")
)

print("Silver and quarantine tables created.")


### Verify that no records were lost ###

In [0]:

bronze_count = spark.table(
    "workspace.bronze.yellow_taxi_trips"
).count()

silver_count = spark.table(
    "workspace.silver.yellow_taxi_trips"
).count()

quarantine_count = spark.table(
    "workspace.silver.yellow_taxi_quarantine"
).count()

print("Bronze records:", bronze_count)
print("Silver accepted:", silver_count)
print("Quarantined:", quarantine_count)

assert bronze_count == silver_count + quarantine_count

print("PASS: All source records accounted for")

# Inspect quarantine reasons
display(
    spark.table("workspace.silver.yellow_taxi_quarantine")
    .groupBy("quality_reason")
    .count()
)



## Silver Layer — Validation Results

### Source
workspace.bronze.yellow_taxi_trips

### Output Tables
- workspace.silver.yellow_taxi_trips
- workspace.silver.yellow_taxi_quarantine

### Validation Rules
1. Missing pickup/drop-off timestamps → QUARANTINE
2. Negative trip duration → QUARANTINE
3. Zero-distance trips → Flagged, retained
4. Negative fares → Flagged, retained

### Verified Results
- Bronze records: 3,475,226
- Accepted records: 3,475,102
- Quarantined records: 124
- Quarantine reason: NEGATIVE_DURATION
- Reconciliation difference: 0

### Current Limitations
- Only the initial critical validation rules are implemented.
- Records with suspicious monetary values are retained.
- Delta tables currently use full overwrite for January.
- Incremental processing and automated orchestration remain pending.
